# 82514 · Sesión S17 — Percepción aprendida: núcleos aprendidos, entrenar y probar, y un detector medido

**Bloque 6** · lunes 16 de noviembre de 2026 · 2 h · IQS Universitat Ramon Llull

Un núcleo y una neurona aprendidos, y dos detectores medidos: el clásico del S17a y NanoDet-Plus.

| | |
|---|---|
| **En clase** | Al acabar el S17a, ejecutad la instalación (descarga NanoDet-Plus). Tras el descanso, ocho minutos: secciones 1 a 4 y el ejercicio de clase. |
| **En casa** | Ejercicios 1 a 3; las soluciones, al final. |
| **Entrega** | Al acabar la clase, individual: este cuaderno y el S17a, ejecutados y con «✎ Vuestra respuesta» escrita (Archivo → Descargar → Descargar .ipynb), a la Tarea «Taller S17: procesado de imagen» de Moodle. Sin nota: presentado o no presentado. |

Cada paso tiene tres celdas: **▶ Parámetros**, lo único que se cambia; **⚙ Cálculo**, que no hace falta tocar; y **✔ Resultado**, lo que sale.

---

In [ ]:
# @title ⚙ Instalación · ejecutad esta celda primero
# Ejecutar una sola vez. En Colab tarda unos veinte segundos.
import importlib, subprocess, sys

def asegurar(mods):
    faltan = []
    for pip_name, import_name in mods:
        try:
            importlib.import_module(import_name)
        except ImportError:
            faltan.append(pip_name)
    if faltan:
        print('Instalando:', ' '.join(faltan))
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + faltan, check=False)
    else:
        print('Todo instalado ya.')

asegurar([('numpy', 'numpy'), ('matplotlib', 'matplotlib')])

import importlib.metadata as md

OPENCV = '4.14.0.94'          # la versión de todo el bloque 6 (cv2.__version__ = '4.14.0')
FIJAS = {'scikit-learn': '1.6.1', 'scikit-image': '0.25.2'}               # las de Colab: se comprueban y, si cambian, se instalan antes de importarlas

def opencv_activo():
    """Versión de OpenCV que se importaría, mirada en otro proceso: Colab trae tres paquetes de OpenCV que
    comparten la carpeta cv2, y lo que dice pip no basta para saber cuál se carga."""
    r = subprocess.run([sys.executable, '-c', 'import cv2; print(cv2.__version__)'], capture_output=True, text=True)
    return r.stdout.strip()

def fijar():
    if opencv_activo() != OPENCV.rsplit('.', 1)[0]:
        print('Instalando OpenCV', OPENCV, '...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps', f'opencv-python-headless=={OPENCV}'], check=False)
    for paquete, version in FIJAS.items():
        try:
            ya = md.version(paquete)
        except md.PackageNotFoundError:
            ya = None
        if ya != version:
            print('Instalando', paquete, version, '...')
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', f'{paquete}=={version}'], check=False)

fijar()
import os, json, time, tempfile, warnings
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import cv2
warnings.filterwarnings('ignore')
EN_COLAB = 'google.colab' in sys.modules
np.set_printoptions(precision=3, suppress=True)
plt.rcParams['figure.figsize'] = (10, 3.6)
IQS_AZUL, IQS_VERDE, GRIS, AMBAR, ROJO = '#1B2A80', '#1FA355', '#8A8FA8', '#B9770E', '#C0392B'
COMA = FuncFormatter(lambda v, p: f'{v:g}'.replace('.', ',').replace('-', '−'))   # ejes con coma decimal

def coma(x, n=2):
    """Un número con coma decimal y signo menos, para imprimir."""
    return f'{x:.{n}f}'.replace('.', ',').replace('-', '−')

def ejes(*axs):
    for ax in axs:
        ax.xaxis.set_major_formatter(COMA); ax.yaxis.set_major_formatter(COMA)

def sin_ejes(*axs):
    for ax in axs:
        ax.set_xticks([]); ax.set_yticks([])

print('OpenCV', cv2.__version__, '· numpy', np.__version__, '· en Colab:', EN_COLAB)
if cv2.__version__ != OPENCV.rsplit('.', 1)[0]:
    print('AVISO: este cuaderno se probó con OpenCV', OPENCV, '. Entorno de ejecución → Reiniciar sesión, y vuelve a ejecutar esta celda.')
if not np.__version__.startswith('2.4'):
    print('AVISO: este cuaderno se probó con numpy 2.4; las cifras con semilla pueden cambiar un poco.')

# El detector: NanoDet-Plus (OpenCV Zoo, Apache 2.0; entrenado con COCO, 80 clases), 3,8 MB, de una revisión fijada.
NANODET = ('https://huggingface.co/opencv/object_detection_nanodet/resolve/5bfd47077350a726ad440dd7bd1e1e35e8ebcfb2/'
           'object_detection_nanodet_2022nov.onnx')
COPIA = ('https://media.githubusercontent.com/media/opencv/opencv_zoo/main/models/object_detection_nanodet/'
         'object_detection_nanodet_2022nov.onnx')
HUELLA = '4b82da9944b88577175ee23a459dce2e26e6e4be573def65b1055dc2d9720186'   # SHA-256 del fichero

def descargar_detector():
    import hashlib, urllib.request
    ruta = os.path.join('.' if EN_COLAB else tempfile.gettempdir(), 'nanodet_2022nov.onnx')
    for k, url in enumerate((NANODET, COPIA), 1):
        try:
            if not (os.path.exists(ruta) and hashlib.sha256(open(ruta, 'rb').read()).hexdigest() == HUELLA):
                datos = urllib.request.urlopen(url, timeout=20).read()
                if hashlib.sha256(datos).hexdigest() != HUELLA:
                    continue
                open(ruta, 'wb').write(datos)
            return ruta
        except Exception as e:
            print(f'No se pudo descargar de la copia {k} ({type(e).__name__}).')
    print('AVISO: sin el detector. La sección 4 imprime lo que habría salido (medido con OpenCV 4.14.0).')
    print('  La copia 2 apunta a la rama principal de OpenCV Zoo: si cambia el fichero, su huella no cuadra y no se usa.')
    return None

RUTA_DETECTOR = descargar_detector()
import sklearn, skimage
print('scikit-learn', sklearn.__version__, '· scikit-image', skimage.__version__, '· detector:', 'listo' if RUTA_DETECTOR else 'no')

## 0. La mesa del puesto de picking

*En clase, sin deteneros*

El simulador del S17a: la mesa con la luz del techo y con una lámpara, y la verdad de cada píxel.

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo · el simulador: ejecutadlo sin abrirlo
# El simulador de la mesa del puesto de picking: la cámara de S16 apuntando a la mesa. Ejecutadlo sin deteneros.
# Cada píxel (submuestreado 2 x 2) es un rayo que sale de la cámara con su K y su distorsión, se gira al mundo con la
# pose de la cámara y se corta con la mesa y con las piezas. Después, desenfoque, ruido y 8 bits, como una foto.
W, H = 640, 480
F = 800.0
K_CURSO = np.array([[F, 0, W/2], [0, F, H/2], [0, 0, 1.0]])
DIST_CURSO = np.array([-0.25, 0.06, 0.0, 0.0, 0.0])
A1, A2 = 0.35, 0.20
TC = np.array([0.30, 0.20, 1.0])
R_0C = np.diag([1.0, -1.0, -1.0])            # Rx(180°): la camara mira hacia abajo
P_PIEZA = np.array([0.40, 0.20])
H_PIEZA, D_PIEZA = 0.010, 0.040

T_0C_NOMINAL = np.eye(4); T_0C_NOMINAL[:3, :3] = R_0C; T_0C_NOMINAL[:3, 3] = TC   # el plano del montaje


def T_C0_verdad():
    M = np.eye(4); M[:3, :3] = R_0C.T; M[:3, 3] = -R_0C.T @ TC
    return M


class Pieza:
    """Prisma recto de altura h: 'disco' (diametro d), 'cuadrado' (lado d) o 'triangulo' (lado d), girado ang."""
    def __init__(self, forma, x, y, d, ang=0.0, h=H_PIEZA, albedo=0.80):
        self.forma, self.c, self.d, self.ang, self.h, self.albedo = forma, np.array([x, y]), d, ang, h, albedo
        if forma == 'cuadrado':
            v = np.array([[-1, -1], [1, -1], [1, 1], [-1, 1]])*d/2
        elif forma == 'triangulo':
            r = d/np.sqrt(3)
            v = np.array([[r*np.cos(a), r*np.sin(a)] for a in (np.pi/2, np.pi/2 + 2*np.pi/3, np.pi/2 + 4*np.pi/3)])
        else:
            v = None
        if v is not None:
            ca, sa = np.cos(ang), np.sin(ang)
            self.v = v @ np.array([[ca, sa], [-sa, ca]]) + self.c

    def dentro(self, X, Y):
        if self.forma == 'disco':
            return (X - self.c[0])**2 + (Y - self.c[1])**2 <= (self.d/2)**2
        ok = np.ones(X.shape, bool)
        v = self.v
        for a, b in zip(v, np.roll(v, -1, axis=0)):      # poligono convexo, vertices en sentido antihorario
            ok &= (b[0] - a[0])*(Y - a[1]) - (b[1] - a[1])*(X - a[0]) >= 0
        return ok

    def area_px(self, f=F, Z=1.0):
        """Area aproximada de la cara de arriba en la imagen (a la distancia Z - h)."""
        e = f/(Z - self.h)
        if self.forma == 'disco':
            return np.pi*(self.d/2*e)**2
        if self.forma == 'cuadrado':
            return (self.d*e)**2
        return np.sqrt(3)/4*(self.d*e)**2


def vetas(X, Y):
    """La veta de la madera de la mesa: rayas onduladas a lo largo de y, cada 8 mm, de un 8 % de amplitud."""
    return 0.08*np.sin(2*np.pi*X/0.008 + 2.5*np.sin(2*np.pi*Y/0.15))


ESCENA = [Pieza('disco', 0.40, 0.20, 0.040),                 # la pieza: en P, el punto de la inversa de S10
          Pieza('cuadrado', 0.12, 0.34, 0.040, ang=0.35),     # dos piezas que no son la nuestra
          Pieza('triangulo', 0.56, 0.40, 0.050, ang=0.2)]


class Mesa:
    """La camara (K y distorsion de la unidad) a 1 m de la mesa, cenital: por defecto, en el plano del montaje,
    T_0C = Trans(0,3; 0,2; 1)·Rx(180°); con T_0C, donde de verdad la dejo su soporte (la unidad de S16)."""
    def __init__(self, K=K_CURSO, dist=DIST_CURSO, T_0C=None, ss=2):
        T = T_0C_NOMINAL if T_0C is None else np.asarray(T_0C, float)
        self.K, self.dist, self.ss = K, dist, ss
        self.R, self.o = T[:3, :3], T[:3, 3]                        # giro y origen de la camara en {0}
        us = (np.arange(W*ss) + 0.5)/ss - 0.5
        vs = (np.arange(H*ss) + 0.5)/ss - 0.5
        U, V = np.meshgrid(us, vs)
        pts = np.stack([U.ravel(), V.ravel()], -1).reshape(-1, 1, 2)
        n = cv2.undistortPointsIter(pts, K, dist, None, None,
                                    (cv2.TERM_CRITERIA_COUNT | cv2.TERM_CRITERIA_EPS, 60, 1e-12)).reshape(-1, 2)
        d = (self.R @ np.vstack([n.T, np.ones(len(n))])).T        # direccion del rayo en el mundo
        self.d = d
        self.X0, self.Y0 = self.corta(0.0)                         # donde corta cada rayo a la mesa

    def corta(self, z):
        lam = (z - self.o[2])/self.d[:, 2]
        return self.o[0] + lam*self.d[:, 0], self.o[1] + lam*self.d[:, 1]

    def foto(self, piezas=ESCENA, rng=None, luz=(1.0, 1.0), mesa=0.35, lateral=0.30, sombra=0.45,
             lampara=None, blur=0.7, ruido=2.0, textura=vetas, mascara=False):
        """luz = (en x = -0,1 m, en x = 0,7 m): la iluminacion de la mesa varia en linea recta a lo ancho.
        lampara: hacia donde esta la lampara, en metros de mesa por metro de altura (None: luz del techo, sin
        sombras; (1,73; 0,5), una lampara a 30° sobre la mesa, a la derecha). Devuelve la imagen
        en gris (uint8) y, si mascara, la verdad: un entero por pixel (0 mesa, i+1 la cara de arriba de la pieza i)."""
        rng = np.random.default_rng(0) if rng is None else rng
        X0, Y0 = self.X0, self.Y0
        L = luz[0] + (luz[1] - luz[0])*np.clip((X0 + 0.10)/0.80, 0, 1)
        alb = np.full(X0.shape, mesa)
        if textura is not None:
            alb = alb*(1 + textura(X0, Y0))
        verdad = np.zeros(X0.shape, np.int32)
        for i, p in enumerate(piezas):
            # solo los rayos que pasan cerca de la pieza (mas rapido: la escena entera son 1,2 millones de rayos)
            alc = p.d + (0 if lampara is None else np.hypot(*lampara)*p.h) + 0.01
            j = np.flatnonzero((X0 - p.c[0])**2 + (Y0 - p.c[1])**2 < alc**2)
            x0, y0 = X0[j], Y0[j]
            base = p.dentro(x0, y0)
            if lampara is not None:      # sombra: desde ese punto de la mesa, la lampara se ve a traves de la pieza
                somb = np.zeros(len(j), bool)
                for z in (0.25*p.h, 0.5*p.h, 0.75*p.h, p.h):
                    somb |= p.dentro(x0 + lampara[0]*z, y0 + lampara[1]*z)
                alb[j] = np.where(somb & ~base, alb[j]*sombra, alb[j])
            lam = (p.h - self.o[2])/self.d[j, 2]
            top = p.dentro(self.o[0] + lam*self.d[j, 0], self.o[1] + lam*self.d[j, 1])
            alb[j] = np.where(base & ~top, lateral, alb[j])     # el lateral: entra por el borde y no sale por la cara
            alb[j] = np.where(top, p.albedo, alb[j])
            verdad[j] = np.where(top, i + 1, verdad[j])
        img = 255*alb*L
        ss = self.ss
        img = img.reshape(H*ss, W*ss).reshape(H, ss, W, ss).mean(axis=(1, 3))
        img = cv2.GaussianBlur(img, (0, 0), blur)
        img = img + rng.normal(0, ruido, img.shape)
        img = np.clip(np.round(img), 0, 255).astype(np.uint8)
        if mascara:     # la verdad en la rejilla de pixeles: la pieza que cubre al menos 2 de los 4 subpixeles
            vs = verdad.reshape(H, ss, W, ss).transpose(0, 2, 1, 3).reshape(H, W, ss*ss)
            v = np.zeros((H, W), np.int32)
            for i in range(1, len(piezas) + 1):
                v[(vs == i).sum(-1)*2 >= ss*ss] = i
            return img, v
        return img


def a_la_mesa(uv, K, dist, T_C0, z=0.0):
    """Del pixel al plano z de {0}: quitar la distorsion, rayo K^-1·p, girarlo al mundo y cortarlo con el plano."""
    uv = np.atleast_2d(np.asarray(uv, float)).reshape(-1, 1, 2)
    n = cv2.undistortPointsIter(uv, K, np.asarray(dist, float), None, None,
                                (cv2.TERM_CRITERIA_COUNT | cv2.TERM_CRITERIA_EPS, 100, 1e-12)).reshape(-1, 2)
    n = np.column_stack([n, np.ones(len(n))])
    T = np.linalg.inv(np.asarray(T_C0, float))
    R, o = T[:3, :3], T[:3, 3]
    d = (R @ n.T).T
    lam = (z - o[2])/d[:, 2]
    return (o + lam[:, None]*d)[:, :2]


def ik2r(x, y, codo=+1):
    """La inversa cerrada del 2R de S10 (a1 = 0,35 m, a2 = 0,2 m); codo = +1, codo abajo."""
    c2 = (x*x + y*y - A1**2 - A2**2)/(2*A1*A2)
    q2 = codo*np.arccos(np.clip(c2, -1, 1))
    q1 = np.arctan2(y, x) - np.arctan2(A2*np.sin(q2), A1 + A2*np.cos(q2))
    return q1, q2


def piezas_en(mask, area_min=300, area_max=3000):
    """Componentes conexas de una mascara binaria: area, centroide (momentos), circularidad y caja."""
    n, et, st, cen = cv2.connectedComponentsWithStats(mask, connectivity=8)
    out = []
    for k in range(1, n):
        a = st[k, cv2.CC_STAT_AREA]
        if a < area_min or a > area_max:
            continue
        m = (et == k).astype(np.uint8)
        M = cv2.moments(m, binaryImage=True)
        cont = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)[0][0]
        per = cv2.arcLength(cont, True)
        x, y, w, h = st[k, :4]
        out.append(dict(area=M['m00'], u=M['m10']/M['m00'], v=M['m01']/M['m00'],
                        circ=4*np.pi*M['m00']/per**2, caja=(x, y, x + w, y + h)))
    return out

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
cam = Mesa()                                          # la cámara del curso, cenital a 1 m
LAMPARA = dict(luz=(0.5, 1.0), lampara=(1.73, 0.5))   # la lámpara: a 30° sobre la mesa, a la derecha
I_t, V_t = cam.foto(mascara=True)                     # con la luz del techo: la foto y la verdad de cada píxel
I_l, V_l = cam.foto(rng=np.random.default_rng(1), mascara=True, **LAMPARA)   # con la lámpara
ZONA_0 = (205, 275, 350, 440)                         # para dibujar: el disco, ampliado (filas v0:v1 y columnas u0:u1 [px])
from matplotlib import patheffects
HALO = [patheffects.withStroke(linewidth=4, foreground='white')]   # para dibujar: un borde blanco, para ver las líneas sobre las fotos

In [ ]:
# @title ✔ Resultado
print('Listo: la mesa, con la luz del techo y con la lámpara.')
zv0, zv1, zu0, zu1 = ZONA_0
fig, axs = plt.subplots(1, 3, figsize=(14, 4.1), gridspec_kw=dict(width_ratios=[4/3, 4/3, (zu1 - zu0)/(zv1 - zv0)]))
for ax, im, t in ((axs[0], I_t, 'Con la luz del techo'), (axs[1], I_l, 'Con la lámpara, a la derecha')):
    ax.imshow(im, cmap='gray', vmin=0, vmax=255); ax.set_title(t); sin_ejes(ax)
axs[1].add_patch(plt.Rectangle((zu0 - 0.5, zv0 - 0.5), zu1 - zu0, zv1 - zv0, fill=False, ec='white', lw=1.2, ls='--'))
axs[2].imshow(I_l[zv0:zv1, zu0:zu1], cmap='gray', vmin=0, vmax=255, extent=(zu0 - 0.5, zu1 - 0.5, zv1 - 0.5, zv0 - 0.5))
axs[2].contour(np.arange(zu0, zu1), np.arange(zv0, zv1), (V_l[zv0:zv1, zu0:zu1] > 0).astype(float), levels=[0.5],
               colors=IQS_AZUL, linewidths=2).set_path_effects(HALO)
axs[2].set_title('Ampliada: la verdad (azul), sin la sombra', fontsize=11)
axs[2].set_xlabel('u [px]'); axs[2].set_ylabel('v [px]')
plt.tight_layout(); plt.show()

## 1. Un núcleo aprendido de ejemplos se parece a Sobel

*En clase, sin deteneros · diapositiva 34*

Mínimos cuadrados: cada píxel con sus 8 vecinos y, como objetivo, la derivada en u de la foto sin ruido. Mirad el núcleo.

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
from numpy.lib.stride_tricks import sliding_window_view
SOBEL = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], float)
In = I_t.astype(float)/255                                          # la foto, con el ruido del sensor
I0 = cam.foto(ruido=0.0).astype(float)/255                          # la misma foto sin ruido: solo un simulador la tiene
A9 = sliding_window_view(np.pad(In, 1, mode='edge'), (3, 3)).reshape(-1, 9)        # cada píxel con sus 8 vecinos
I0p = np.pad(I0, 1, mode='edge')
objetivo = ((I0p[1:-1, 2:] - I0p[1:-1, :-2])/2).ravel()            # lo que debería salir: la derivada en u, sin ruido
w9 = np.linalg.lstsq(A9, objetivo, rcond=None)[0].reshape(3, 3)     # los 9 pesos que menos se equivocan
CORR_SOBEL = np.corrcoef(w9.ravel(), SOBEL.ravel())[0, 1]           # su correlación con Sobel en u
ERRORES = []                                                        # (núcleo, cuánto se aparta de la derivada sin ruido [niveles por píxel])
FILTRADAS = {}                                                      # para dibujar: la foto filtrada con cada núcleo [niveles por píxel]
for nom, k in (('la diferencia centrada, sin promediar', np.array([[0, 0, 0], [-0.5, 0, 0.5], [0, 0, 0]])), ('Sobel/8', SOBEL/8), ('el aprendido', w9)):
    o = cv2.filter2D(In, -1, k, borderType=cv2.BORDER_REPLICATE).ravel()
    ERRORES.append((nom, 255*np.sqrt(np.mean((o - objetivo)**2))))
    FILTRADAS[nom] = 255*o.reshape(In.shape)
OBJETIVO = 255*objetivo.reshape(In.shape)                           # para dibujar: la derivada sin ruido [niveles por píxel]
BORDE, FILA = (222, 259, 374, 428), 240                             # para dibujar: el recorte del disco (filas v0:v1, columnas u0:u1) y una fila

In [ ]:
# @title ✔ Resultado
print(A9.shape[0], 'ejemplos y 9 incógnitas')
print('el núcleo aprendido, por 8:')
for fila in 8*w9:
    print('  ', '  '.join(f'{coma(np.round(x, 2) + 0.0):>6s}' for x in fila))
print('correlación con Sobel en u:', coma(CORR_SOBEL))
for nom, e in ERRORES:
    print(f'{nom}: se aparta de la derivada sin ruido {coma(e)} niveles por píxel')
fig, axs = plt.subplots(1, 2, figsize=(8, 3.9))
for ax, k, t in ((axs[0], 8*w9, 'El aprendido (× 8)'), (axs[1], SOBEL, 'Sobel en u')):
    ax.imshow(k, cmap='RdBu_r', vmin=-2.6, vmax=2.6); ax.set_title(t); sin_ejes(ax)
    for (i, j), x in np.ndenumerate(k):
        ax.text(j, i, coma(np.round(x, 2) + 0.0), ha='center', va='center', fontsize=13, fontweight='bold')
plt.suptitle(f'Los nueve pesos: correlación con Sobel en u, {coma(CORR_SOBEL)}'); plt.tight_layout(); plt.show()

# la foto filtrada, ampliada al disco: el objetivo, el aprendido y Sobel/8, y una fila con los tres núcleos
bv0, bv1, bu0, bu1 = BORDE
fig = plt.figure(figsize=(15, 6.6))
gs = fig.add_gridspec(2, 4, width_ratios=[1, 1, 1, 0.05], height_ratios=[1, 0.85], hspace=0.45, wspace=0.25)
for k, (t, im) in enumerate((('El objetivo: la derivada sin ruido', OBJETIVO), ('Con el aprendido', FILTRADAS['el aprendido']),
                             ('Con Sobel/8', FILTRADAS['Sobel/8']))):
    ax = fig.add_subplot(gs[0, k])
    h = ax.imshow(im[bv0:bv1, bu0:bu1], cmap='RdBu_r', vmin=-50, vmax=50, extent=(bu0 - 0.5, bu1 - 0.5, bv1 - 0.5, bv0 - 0.5))
    ax.axhline(FILA, color='black', lw=1, ls='--'); ax.set_title(t + ' (ampliada)', fontsize=11); ax.set_xlabel('u [px]')
    if k == 0:
        ax.set_ylabel('v [px]')
fig.colorbar(h, cax=fig.add_subplot(gs[0, 3]), label='niveles por píxel')
u = np.arange(bu0, bu1)
COLOR_NUCLEO = {'la diferencia centrada, sin promediar': GRIS, 'Sobel/8': AMBAR, 'el aprendido': IQS_VERDE}
ax = fig.add_subplot(gs[1, :2])
ax.plot(u, OBJETIVO[FILA, bu0:bu1], color=IQS_AZUL, lw=4, alpha=0.5, label='el objetivo')
for nom, e in ERRORES:
    ax.plot(u, FILTRADAS[nom][FILA, bu0:bu1], '-', color=COLOR_NUCLEO[nom], lw=1.4, label=nom)
ax.set_title(f'La fila v = {FILA} (la discontinua)', fontsize=11); ax.set_xlabel('u [px]'); ax.set_ylabel('niveles por píxel')
ax.grid(True); ax.legend(fontsize=9, loc='lower center'); ejes(ax)
ax = fig.add_subplot(gs[1, 2:])
for nom, e in ERRORES:
    ax.plot(u, FILTRADAS[nom][FILA, bu0:bu1] - OBJETIVO[FILA, bu0:bu1], '-', color=COLOR_NUCLEO[nom], lw=1.6,
            label=f'{nom}: {coma(e)} en toda la foto')
ax.axhline(0, color=IQS_AZUL, lw=1)
ax.set_title('Lo que se aparta del objetivo en esa fila', fontsize=11); ax.set_xlabel('u [px]'); ax.set_ylabel('niveles por píxel')
ax.set_ylim(-11, 4.5); ax.grid(True); ax.legend(fontsize=9, loc='lower center', ncol=1); ejes(ax)
plt.show()

**Qué tiene que salir:** 307 200 ejemplos y 9 incógnitas; correlación con Sobel 0,96; en la figura, el aprendido resta la columna izquierda de la derecha, como Sobel, y en la fila del disco se aparta del objetivo menos que la diferencia centrada.

## 2. Acertar al entrenar no es acertar

*En clase, sin deteneros · diapositiva 36 · unos treinta segundos en Colab*

¿Pieza o mesa?, píxel a píxel: una neurona entrenada con 4 escenas y probada con 20 nuevas por luz. Mirad la otra luz.

In [ ]:
# @title ▶ Parámetros · cambiad solo esto
# el vecindario que mira la neurona: su lado [px], impar (7: 49 pesos, un núcleo de 7 x 7)
VECINDARIO = 7  # @param {type:"number"}

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
from sklearn.linear_model import LogisticRegression
K7 = int(VECINDARIO)

def escena_azar(rng, n=6):
    ps = []
    while len(ps) < n:
        f = rng.choice(['disco', 'cuadrado', 'triangulo']); x = rng.uniform(-0.05, 0.65); y = rng.uniform(-0.05, 0.45)
        if all(np.hypot(x - p.c[0], y - p.c[1]) > 0.07 for p in ps):
            ps.append(Pieza(f, x, y, rng.uniform(0.03, 0.05), ang=rng.uniform(0, np.pi)))
    return ps

def muestras(rng, n_esc, n_pix, luz='techo'):
    """n_pix píxeles por escena, mitad pieza y mitad mesa: cada uno, su vecindario de 7 x 7, y si es pieza."""
    X, y = [], []
    for e in range(n_esc):
        ps = escena_azar(rng)
        l = luz if luz != 'mezcla' else ('techo' if e % 2 == 0 else 'lampara')
        I_, V_ = cam.foto(ps, rng, mascara=True) if l == 'techo' else cam.foto(ps, rng, mascara=True, **LAMPARA)
        ventanas = sliding_window_view(np.pad(I_.astype(np.float32)/255, K7//2, mode='edge'), (K7, K7))
        es = V_ > 0
        pos, neg = np.argwhere(es), np.argwhere(~es)
        sel = np.vstack([pos[rng.choice(len(pos), n_pix//2)], neg[rng.choice(len(neg), n_pix//2)]])
        X.append(ventanas[sel[:, 0], sel[:, 1]].reshape(len(sel), -1)); y.append(es[sel[:, 0], sel[:, 1]])
    return np.vstack(X), np.concatenate(y)

t0 = time.time()
rng = np.random.default_rng(17)
X_tr, y_tr = muestras(rng, 4, 500, 'techo')        # entrenamiento: 4 escenas con la luz del techo
X_mz, y_mz = muestras(rng, 4, 500, 'mezcla')       # entrenamiento: 4 escenas, dos con cada luz
X_te, y_te = muestras(rng, 20, 100, 'techo')       # prueba: 20 escenas nuevas con la luz del techo
X_la, y_la = muestras(rng, 20, 100, 'lampara')     # prueba: 20 escenas nuevas con la lámpara
T_CONJUNTOS = time.time() - t0                     # lo que tarda en fabricarlos [s]

m1 = LogisticRegression(max_iter=3000).fit(X_tr, y_tr)        # entrenada con la luz del techo
ACIERTO_M1 = [100*m1.score(X, y) for X, y in ((X_tr, y_tr), (X_te, y_te), (X_la, y_la))]   # [%] entrenamiento, techo, lámpara
m2 = LogisticRegression(max_iter=3000).fit(X_mz, y_mz)        # entrenada con las dos luces
ACIERTO_M2 = [100*m2.score(X, y) for X, y in ((X_te, y_te), (X_la, y_la))]                 # [%] techo, lámpara
ACIERTO_M2_TR = 100*m2.score(X_mz, y_mz)                      # [%] para dibujar: la de las dos luces, en su entrenamiento

# para dibujar: pieza o mesa, píxel a píxel, según cada neurona, en una escena nueva con la lámpara (con su propio
# generador, para no tocar rng)
rng_mapa = np.random.default_rng(1700)
I_mapa, V_mapa = cam.foto(escena_azar(rng_mapa), rng_mapa, mascara=True, **LAMPARA)
MAPAS = [cv2.filter2D(I_mapa.astype(np.float32)/255, -1, m.coef_.reshape(K7, K7).astype(np.float32),
                      borderType=cv2.BORDER_REPLICATE) + m.intercept_[0] > 0 for m in (m1, m2)]   # True: pieza (lo mismo que m.predict)
MAPA_VERDAD = V_mapa > 0

In [ ]:
# @title ✔ Resultado
print(f'{len(X_tr)} píxeles por conjunto ({coma(T_CONJUNTOS, 0)} s)')
print(f'Entrenada con el techo: entrenamiento {coma(ACIERTO_M1[0], 1)} %; prueba, misma luz {coma(ACIERTO_M1[1], 1)} %;'
      f' con la lámpara {coma(ACIERTO_M1[2], 1)} %')
print(f'Entrenada con las dos luces: con el techo {coma(ACIERTO_M2[0], 1)} %; con la lámpara {coma(ACIERTO_M2[1], 1)} %')
from matplotlib.patches import Patch
from matplotlib.colors import to_rgb
fig, ax = plt.subplots(figsize=(10, 3.8))
grupos = (('entrenada con el techo', ACIERTO_M1), ('entrenada con las dos luces', [ACIERTO_M2_TR, *ACIERTO_M2]))
for j, (t, c) in enumerate((('entrenamiento: sus ejemplos', IQS_AZUL), ('prueba con la luz del techo', IQS_VERDE),
                            ('prueba con la lámpara', AMBAR))):
    vals = [g[1][j] for g in grupos]
    barras = ax.bar(np.arange(2) + (j - 1)*0.27, vals, 0.25, color=c, label=t)
    for b, v in zip(barras, vals):
        ax.text(b.get_x() + b.get_width()/2, v - 0.8, coma(v, 1), ha='center', va='top', color='white', fontsize=10, fontweight='bold')
ax.set_xticks(range(2)); ax.set_xticklabels([g[0] for g in grupos]); ax.set_ylim(70, 101); ax.set_ylabel('acierto [%]')
ax.set_title('El dominio: la prueba, con 20 escenas nuevas por luz'); ax.grid(True, axis='y'); ax.yaxis.set_major_formatter(COMA)
ax.legend(loc='center left', bbox_to_anchor=(1.01, 0.5), fontsize=9.5); plt.tight_layout(); plt.show()

# pieza o mesa, píxel a píxel, en una escena nueva con la lámpara: lo que dice cada neurona frente a la verdad
fondo = np.repeat((0.45 + 0.55*I_mapa/255)[..., None], 3, axis=-1)                 # la foto, aclarada
fig, axs = plt.subplots(1, 3, figsize=(15, 4.6))
axs[0].imshow(I_mapa, cmap='gray', vmin=0, vmax=255); axs[0].set_title('Una escena nueva, con la lámpara')
for ax, mapa, t in zip(axs[1:], MAPAS, ('Entrenada con el techo', 'Entrenada con las dos luces')):
    rgb = fondo.copy()
    rgb[mapa & MAPA_VERDAD] = to_rgb(IQS_VERDE); rgb[mapa & ~MAPA_VERDAD] = to_rgb(ROJO); rgb[~mapa & MAPA_VERDAD] = to_rgb(IQS_AZUL)
    ax.imshow(rgb); ax.set_title(t)
sin_ejes(*axs)
fig.legend(handles=[Patch(color=IQS_VERDE, label='dice pieza, y lo es'), Patch(color=ROJO, label='dice pieza, y es mesa'),
                    Patch(color=IQS_AZUL, label='dice mesa, y es pieza')], loc='lower center', ncol=3, fontsize=10)
plt.tight_layout(rect=(0, 0.08, 1, 1)); plt.show()

**Qué tiene que salir:** entrenada con el techo, 99,8 % y, con la lámpara, 83,7 %; con las dos luces, 96,0 %; en la figura, en la escena nueva con la lámpara, la entrenada con el techo pierde (en azul) las piezas de la zona oscura.

### Pocos ejemplos

Con la luz del techo y la media de varios sorteos: mirad el entrenamiento frente a la prueba.

In [ ]:
# @title ▶ Parámetros · cambiad solo esto
# los números de ejemplos de entrenamiento que se prueban
N_EJEMPLOS = (6, 10, 20, 50, 200, 1000)  # @param {type:"raw"}
# los sorteos de ejemplos para cada número: se da la media
SORTEOS = 5  # @param {type:"number"}

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
rng_s = np.random.default_rng(170)
POCOS = []                                         # (ejemplos, acierto medio en el entrenamiento y en la prueba [%])
for n in N_EJEMPLOS:
    n, acc = int(n), []
    for rep in range(int(SORTEOS)):
        pos_, neg_ = np.flatnonzero(y_tr), np.flatnonzero(~y_tr)
        idx = np.concatenate([rng_s.choice(pos_, n//2, replace=False), rng_s.choice(neg_, n - n//2, replace=False)])
        mm = LogisticRegression(max_iter=3000, C=1e4).fit(X_tr[idx], y_tr[idx])
        acc.append((mm.score(X_tr[idx], y_tr[idx]), mm.score(X_te, y_te)))
    acc = 100*np.array(acc)
    POCOS.append((n, acc[:, 0].mean(), acc[:, 1].mean()))

In [ ]:
# @title ✔ Resultado
print(f'Pocos ejemplos (con la luz del techo; media de {int(SORTEOS)} sorteos; prueba: las 20 escenas nuevas):')
for n, a_tr, a_te in POCOS:
    print(f'  {n:5d} ejemplos: entrenamiento {coma(a_tr, 1)} %; prueba {coma(a_te, 1)} %')
fig, ax = plt.subplots(figsize=(8, 3.8))
n_, a_tr_, a_te_ = [p[0] for p in POCOS], [p[1] for p in POCOS], [p[2] for p in POCOS]
ax.fill_between(n_, a_te_, a_tr_, color=ROJO, alpha=0.12, label='lo que acierta solo con los suyos')
ax.semilogx(n_, a_tr_, 'o-', color=IQS_AZUL, lw=2, label='entrenamiento')
ax.semilogx(n_, a_te_, 's-', color=IQS_VERDE, lw=2, label='prueba: 20 escenas nuevas')
ax.set_xticks(n_); ax.set_xticks([], minor=True)
ax.set_xlabel('ejemplos de entrenamiento (eje logarítmico)'); ax.set_ylabel('acierto [%]')
ax.set_title(f'Sobreajuste: luz del techo, media de {int(SORTEOS)} sorteos'); ax.grid(True); ax.legend(fontsize=9, loc='lower right')
ejes(ax); plt.tight_layout(); plt.show()

**Qué tiene que salir:** con 6 ejemplos, 100 % y 95,1 %: los memoriza; con 1000, 100 % y 99,8 %; en la figura, la banda roja se cierra con más ejemplos.

## 3. El detector clásico del S17a, medido

*En clase · diapositivas 38 y 40*

El detector de discos del S17a en 20 escenas con verdad conocida, la mitad con la lámpara. Mirad la precisión y la exhaustividad.

In [ ]:
# @title ▶ Parámetros · cambiad solo esto
# el umbral adaptativo del S17a: el lado de la ventana [px, impar] y cuánto tiene que pasar un píxel de su media [niveles]
VENTANA = 51  # @param {type:"number"}
MAS = 10  # @param {type:"number"}
# el área de un disco en la imagen, mínima y máxima [px]
AREA_DISCO = (500, 1200)  # @param {type:"raw"}
# el IoU con la caja de un disco verdadero a partir del cual una caja es un acierto
IOU_MIN = 0.5  # @param {type:"number"}
# las circularidades mínimas que se miden
CIRC_MIN = (0.75, 0.94)  # @param {type:"raw"}

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
def caja_de(V, k):
    ij = np.argwhere(V == k); return (ij[:, 1].min(), ij[:, 0].min(), ij[:, 1].max() + 1, ij[:, 0].max() + 1)

def iou(A, B):
    ix = max(0, min(A[2], B[2]) - max(A[0], B[0])); iy = max(0, min(A[3], B[3]) - max(A[1], B[1]))
    i = ix*iy
    return i/((A[2] - A[0])*(A[3] - A[1]) + (B[2] - B[0])*(B[3] - B[1]) - i)

IOU_EJEMPLO = iou((100, 100, 180, 160), (120, 110, 200, 170))     # dos cajas de ejemplo
CAJAS_EJEMPLO = ((100, 100, 180, 160), (120, 110, 200, 170))       # para dibujar: la verdadera y la detectada

def escena_det(rng):
    ps = []
    for f in ['disco']*rng.integers(1, 4) + list(rng.choice(['cuadrado', 'triangulo', 'disco_peq'], rng.integers(2, 5))):
        for _ in range(200):
            x, y = rng.uniform(-0.07, 0.67), rng.uniform(-0.07, 0.47)
            if all(np.hypot(x - p.c[0], y - p.c[1]) > 0.065 for p in ps):
                break
        if f == 'disco_peq':
            ps.append(Pieza('disco', x, y, 0.022))                    # un disco pequeño, de 22 mm: no es la pieza
        elif f == 'disco':
            ps.append(Pieza('disco', x, y, 0.040))
        else:
            ps.append(Pieza(f, x, y, rng.uniform(0.035, 0.05), ang=rng.uniform(0, np.pi)))
    return ps

def detector_clasico(I_):
    """Las cajas y circularidades de lo que pasa el área, tras el umbral adaptativo y la apertura de 3 x 3."""
    b = cv2.adaptiveThreshold(I_, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY, int(VENTANA), -MAS)
    b = cv2.morphologyEx(b, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
    return [(p['caja'], p['circ']) for p in piezas_en(b, area_min=AREA_DISCO[0], area_max=AREA_DISCO[1])]

FOTOS_DET = globals().get('FOTOS_DET', {})   # para dibujar: semilla -> las fotos de sus escenas (no se vacía al repetir la celda)

def fabrica(semilla, n):
    """n escenas, la mitad con la lámpara: lo que da el detector (cajas y circularidades) y las cajas de los discos verdaderos.
    Guarda las fotos en FOTOS_DET[semilla], para dibujarlas."""
    rng_ = np.random.default_rng(semilla); esc = []; FOTOS_DET[semilla] = []
    for k in range(n):
        ps = escena_det(rng_)
        I_, V_ = cam.foto(ps, rng_, mascara=True, **(LAMPARA if k % 2 else {}))
        verdad = [caja_de(V_, i + 1) for i, p in enumerate(ps) if p.forma == 'disco' and p.d > 0.03 and (V_ == i + 1).sum() > 0]
        esc.append((detector_clasico(I_), verdad)); FOTOS_DET[semilla].append(I_)
    return esc

t0 = time.time()
escenas = fabrica(19, 20)                          # las 20 escenas con las que se elige el umbral
T_ESCENAS = time.time() - t0                       # lo que tarda [s]
N_DISCOS = sum(len(v) for _, v in escenas)

def mide(circ_min, escenas=escenas):
    """VP, FP y FN del detector con una circularidad mínima."""
    VP = FP = FN = 0
    for dets, verdad in escenas:
        usadas = set()
        for caja, circ in dets:
            if circ < circ_min:
                continue
            io = [(iou(caja, v), j) for j, v in enumerate(verdad) if j not in usadas]
            if io and max(io)[0] >= IOU_MIN:
                usadas.add(max(io)[1]); VP += 1
            else:
                FP += 1
        FN += len(verdad) - len(usadas)
    return VP, FP, FN

def medidas(circ_min, esc):
    """Precisión, VP/(VP + FP), y exhaustividad, VP/(VP + FN); sin cajas, la precisión es 1."""
    VP, FP, FN = mide(circ_min, esc)
    return (VP/(VP + FP) if VP + FP else 1.0), VP/(VP + FN)

SECCION3 = [(cm, *mide(cm), *medidas(cm, escenas)) for cm in CIRC_MIN]   # (circularidad, VP, FP, FN, precisión, exhaustividad)

def clasifica(dets, verdad, circ_min):
    """Para dibujar: lo mismo que mide, caja a caja, en una escena: los aciertos y las cajas falsas, (caja, circularidad),
    y los discos perdidos, (caja verdadera, circularidad de lo que el detector vio allí, o None si no vio nada)."""
    usadas, aciertos, falsas = set(), [], []
    for caja, circ in dets:
        if circ < circ_min:
            continue
        io = [(iou(caja, v), j) for j, v in enumerate(verdad) if j not in usadas]
        if io and max(io)[0] >= IOU_MIN:
            usadas.add(max(io)[1]); aciertos.append((caja, circ))
        else:
            falsas.append((caja, circ))
    perdidos = []
    for j, v in enumerate(verdad):
        if j not in usadas:
            vio = [(iou(caja, v), circ) for caja, circ in dets if iou(caja, v) >= IOU_MIN]
            perdidos.append((v, max(vio)[1] if vio else None))
    return aciertos, falsas, perdidos

def errores(dets, verdad):
    """Para dibujar: si hay cajas falsas con la menor circularidad mínima, y discos perdidos con la mayor."""
    return bool(clasifica(dets, verdad, min(CIRC_MIN))[1]) + bool(clasifica(dets, verdad, max(CIRC_MIN))[2])

ESC_DIBUJO = sorted(range(len(escenas)), key=lambda k: -errores(*escenas[k]))[:2]   # para dibujar: las dos primeras con más errores
FOTOS_DIBUJO = [FOTOS_DET[19][k] for k in ESC_DIBUJO]

def dibuja_caja(ax, c, color, texto=None, ls='-'):
    """Para dibujar: una caja (x0, y0, x1, y1) [px] sobre una foto, con su texto debajo."""
    ax.add_patch(plt.Rectangle((c[0] - 0.5, c[1] - 0.5), c[2] - c[0], c[3] - c[1], fill=False, ec=color, lw=2, ls=ls, path_effects=HALO))
    if texto:
        ax.text(c[0] - 0.5, c[3] + 3, texto, color='white', fontsize=9, va='top', bbox=dict(fc=color, ec='none', pad=1))

In [ ]:
# @title ✔ Resultado
print('IoU de (100; 100)-(180; 160) con (120; 110)-(200; 170):', coma(IOU_EJEMPLO, 3))
print(f'20 escenas, {N_DISCOS} discos ({coma(T_ESCENAS, 0)} s)')
for cm, VP, FP, FN, P, R in SECCION3:
    print(f'circularidad ≥ {coma(cm)}: VP {VP}, FP {FP}, FN {FN}; precisión {coma(P, 3)}; exhaustividad {coma(R, 3)}')

# el IoU de las dos cajas de ejemplo
(A_, B_) = CAJAS_EJEMPLO
inter_ = (max(A_[0], B_[0]), max(A_[1], B_[1]), min(A_[2], B_[2]), min(A_[3], B_[3]))     # su intersección
area_ = lambda c: (c[2] - c[0])*(c[3] - c[1])
fig, ax = plt.subplots(figsize=(7, 4.2))
for c, col, t, xy, al in ((A_, IQS_AZUL, 'la verdadera', (A_[0], A_[1] - 2), dict(va='bottom')),
                          (B_, IQS_VERDE, 'la detectada', (B_[2], B_[3] + 2), dict(va='top', ha='right'))):
    ax.add_patch(plt.Rectangle(c[:2], c[2] - c[0], c[3] - c[1], fc=col, alpha=0.15, ec=col, lw=2.4))
    ax.text(*xy, f'{t} ({c[0]}; {c[1]})–({c[2]}; {c[3]})', color=col, fontsize=10, **al)
ax.add_patch(plt.Rectangle(inter_[:2], inter_[2] - inter_[0], inter_[3] - inter_[1], fill=False, ec=AMBAR, lw=2.4, hatch='//'))
ax.text((inter_[0] + inter_[2])/2, (inter_[1] + inter_[3])/2, f'{inter_[2] - inter_[0]} × {inter_[3] - inter_[1]}\n= {area_(inter_)}', color=AMBAR, fontsize=12,
        ha='center', va='center', fontweight='bold', bbox=dict(fc='white', ec='none', alpha=0.8))
ax.set_xlim(90, 210); ax.set_ylim(182, 85); ax.set_aspect('equal'); ax.set_xlabel('u [px]'); ax.set_ylabel('v [px]')
ax.set_title(f'IoU = {area_(inter_)} / ({area_(A_)} + {area_(B_)} − {area_(inter_)}) = {coma(IOU_EJEMPLO, 3)}', fontsize=11)
plt.tight_layout(); plt.show()

# dos de las 20 escenas, con la menor y con la mayor circularidad mínima
fig, axs = plt.subplots(len(ESC_DIBUJO), len(CIRC_MIN), figsize=(min(15, 6*len(CIRC_MIN)), 4.6*len(ESC_DIBUJO) + 0.6), squeeze=False)
for fila, (k, foto) in enumerate(zip(ESC_DIBUJO, FOTOS_DIBUJO)):
    dets, verdad = escenas[k]
    for ax, cm in zip(axs[fila], CIRC_MIN):
        aciertos, falsas, perdidos = clasifica(dets, verdad, cm)
        ax.imshow(foto, cmap='gray', vmin=0, vmax=255); sin_ejes(ax)
        for c, circ in aciertos:
            dibuja_caja(ax, c, IQS_VERDE, coma(circ, 3))
        for c, circ in falsas:
            dibuja_caja(ax, c, ROJO, coma(circ, 3))
        for c, circ in perdidos:
            dibuja_caja(ax, c, IQS_AZUL, 'perdido' + ('' if circ is None else f': {coma(circ, 3)}'), ls='--')
        ax.set_title(f'Escena {k + 1} ({"lámpara" if k % 2 else "techo"}), circularidad ≥ {coma(cm)}:'
                     f' {len(aciertos)} VP, {len(falsas)} FP, {len(perdidos)} FN', fontsize=11)
fig.legend(handles=[plt.Line2D([], [], color=IQS_VERDE, lw=2.5, label='acierto (VP)'),
                    plt.Line2D([], [], color=ROJO, lw=2.5, label='caja falsa (FP)'),
                    plt.Line2D([], [], color=IQS_AZUL, lw=2.5, ls='--', label='disco perdido (FN)')],
           loc='lower center', ncol=3, fontsize=10, title='con la circularidad que mide cada una', title_fontsize=10)
plt.tight_layout(rect=(0, 0.07, 1, 1)); plt.show()

**Qué tiene que salir:** 40 discos; con 0,75, precisión 0,678 y exhaustividad 1; con 0,94, precisión 1 y exhaustividad 0,8; en la figura, con 0,75 entran cuadrados (en rojo) y con 0,94 se pierden discos que miden poco menos (en azul).

### Ejercicio de clase

*En clase; se pone en común · diapositiva 40*

Elegid con las 20 escenas una circularidad mínima que dé las dos a 1, y medidla en 40 nuevas: ¿siguen en 1? ¿Qué valor elegiríais, y por qué?

In [ ]:
# @title ▶ Parámetros · cambiad solo esto
# las circularidades mínimas que se prueban con las 20 escenas
CIRC_PRUEBA = (0.80, 0.85, 0.88, 0.90, 0.92)  # @param {type:"raw"}

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
TABLA_20 = [(cm, *medidas(cm, escenas)) for cm in CIRC_PRUEBA]    # (circularidad, precisión, exhaustividad)
CMS = np.round(np.arange(0.60, 0.981, 0.01), 2)                    # las circularidades de las curvas
CURVA_20 = np.array([medidas(cm, escenas) for cm in CMS])          # precisión y exhaustividad en las 20 escenas
BANDA_20 = CMS[(CURVA_20 == 1).all(axis=1)]                        # para dibujar: donde las dos son 1 en las 20

In [ ]:
# @title ✔ Resultado
print('En las 20 escenas:')
for cm, P, R in TABLA_20:
    print(f'  circularidad ≥ {coma(cm)}: precisión {coma(P, 3)}, exhaustividad {coma(R, 3)}')
fig, ax = plt.subplots(figsize=(9, 4))
if len(BANDA_20):
    ax.axvspan(BANDA_20.min(), BANDA_20.max(), color=IQS_VERDE, alpha=0.12, label='las dos a 1')
ax.plot(CMS, CURVA_20[:, 0], '-', color=IQS_AZUL, lw=2, label='precisión, 20 escenas')
ax.plot(CMS, CURVA_20[:, 1], '-', color=AMBAR, lw=2, label='exhaustividad, 20 escenas')
for cm, VP, FP, FN, P, R in SECCION3:                              # las dos circularidades de la sección 3
    ax.axvline(cm, color=GRIS, lw=1, ls=':')
    ax.plot([cm, cm], [P, R], 'o', color='black', ms=6, zorder=3)
    ax.text(cm, 1.035, coma(cm), ha='center', va='bottom', fontsize=9, color='black', bbox=dict(fc='white', ec='none', pad=0.5))
ax.plot([], [], 'o', color='black', ms=6, label='la sección 3')
ax.set_xlabel('circularidad mínima'); ax.set_ylabel('precisión y exhaustividad'); ax.set_ylim(0.5, 1.07); ax.grid(True)
ax.set_title('El detector clásico en las 20 escenas'); ax.legend(fontsize=9, loc='center left', bbox_to_anchor=(1.01, 0.5))
ejes(ax); plt.tight_layout(); plt.show()

**En 40 escenas nuevas:** poned la vuestra en `CIRC_ELEGIDA` (unos quince segundos en Colab).

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
t0 = time.time()
prueba = fabrica(2020, 40)                         # 40 escenas nuevas, para comprobar: se fabrican una vez
T_PRUEBA = time.time() - t0                        # lo que tarda [s]
N_DISCOS_40 = sum(len(v) for _, v in prueba)

In [ ]:
# @title ▶ Parámetros · cambiad solo esto
# la circularidad mínima que habéis elegido con las 20 escenas
CIRC_ELEGIDA = 0.85  # @param {type:"number"}

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
TABLA_40 =[(cm, *medidas(cm, escenas), *medidas(cm, prueba)) for cm in CIRC_PRUEBA]   # en las 20 y en las 40 nuevas
ELEGIDA = (*medidas(CIRC_ELEGIDA, escenas), *medidas(CIRC_ELEGIDA, prueba))
CURVA_40 = np.array([medidas(cm, prueba) for cm in CMS])          # precisión y exhaustividad en las 40 nuevas
ESC_40 = next((k for k, (d, v) in enumerate(prueba) if any(clasifica(d, v, CIRC_ELEGIDA)[1:])), None)   # para dibujar: la primera
                                                                   # de las 40 con alguna caja falsa o algún disco perdido (None: ninguna)
CAJA_40 = RECORTE_40 = None                                        # para dibujar, ampliado: el primer error de esa escena
if ESC_40 is not None:
    r40_ac, r40_fp, r40_fn = clasifica(*prueba[ESC_40], CIRC_ELEGIDA)
    CAJA_40 = (*r40_fp[0], True) if r40_fp else (*r40_fn[0], False)    # (caja, circularidad, es una caja falsa)
    r40_c, r40_f = CAJA_40[0], FOTOS_DET[2020][ESC_40]
    r40_u0, r40_v0 = max(r40_c[0] - 15, 0), max(r40_c[1] - 15, 0)      # 15 px alrededor de la caja
    r40_u1, r40_v1 = min(r40_c[2] + 15, r40_f.shape[1]), min(r40_c[3] + 15, r40_f.shape[0])
    RECORTE_40 = (r40_f[r40_v0:r40_v1, r40_u0:r40_u1], (r40_u0 - 0.5, r40_u1 - 0.5, r40_v1 - 0.5, r40_v0 - 0.5))   # (recorte, extent [px])

In [ ]:
# @title ✔ Resultado
print(f'40 escenas nuevas, {N_DISCOS_40} discos ({coma(T_PRUEBA, 0)} s)')
for cm, P20, R20, P40, R40 in TABLA_40:
    print(f'circularidad ≥ {coma(cm)}: en las 20, precisión {coma(P20, 3)}, exhaustividad {coma(R20, 3)} ;'
          f' en las 40 nuevas, precisión {coma(P40, 3)}, exhaustividad {coma(R40, 3)}')
print(f'\nLa vuestra, ≥ {coma(CIRC_ELEGIDA)}: en las 20, precisión {coma(ELEGIDA[0], 3)}, exhaustividad {coma(ELEGIDA[1], 3)} ;'
      f' en las 40 nuevas, precisión {coma(ELEGIDA[2], 3)}, exhaustividad {coma(ELEGIDA[3], 3)}')
fig = plt.figure(figsize=(15, 5.4))
gs = fig.add_gridspec(1, 3, width_ratios=[1.25, 0.8, 0.5], wspace=0.2, left=0.06, right=0.98, top=0.9, bottom=0.27)
ax = fig.add_subplot(gs[0])
if len(BANDA_20):
    ax.axvspan(BANDA_20.min(), BANDA_20.max(), color=IQS_VERDE, alpha=0.12, label='las dos a 1, en las 20')
for curva, ls, t in ((CURVA_20, '-', '20 escenas'), (CURVA_40, '--', '40 nuevas')):
    ax.plot(CMS, curva[:, 0], ls, color=IQS_AZUL, lw=2, label=f'precisión, {t}')
    ax.plot(CMS, curva[:, 1], ls, color=AMBAR, lw=2, label=f'exhaustividad, {t}')
ax.axvline(CIRC_ELEGIDA, color='black', lw=1.2, ls='-.', label=f'la vuestra: {coma(CIRC_ELEGIDA)}')
ax.plot([CIRC_ELEGIDA]*4, ELEGIDA, 'o', color='black', ms=5, zorder=3)
ax.set_xlabel('circularidad mínima'); ax.set_ylabel('precisión y exhaustividad'); ax.grid(True)
ax.set_ylim(min(0.5, min(ELEGIDA) - 0.05), 1.03)                   # que se vean los cuatro puntos de la vuestra
ax.legend(fontsize=9, loc='upper center', bbox_to_anchor=(0.5, -0.14), ncol=3); ejes(ax)
ax.set_title(f'En las 20 y en las 40 nuevas; la vuestra, ≥ {coma(CIRC_ELEGIDA)}')
if ESC_40 is None:
    ax = fig.add_subplot(gs[1:]); sin_ejes(ax)
    ax.text(0.5, 0.5, f'Con ≥ {coma(CIRC_ELEGIDA)}, en las 40 nuevas:\nni cajas falsas ni discos perdidos', ha='center', va='center',
            fontsize=12, transform=ax.transAxes)
else:
    ax = fig.add_subplot(gs[1]); sin_ejes(ax)
    aciertos, falsas, perdidos = clasifica(*prueba[ESC_40], CIRC_ELEGIDA)
    ax.imshow(FOTOS_DET[2020][ESC_40], cmap='gray', vmin=0, vmax=255)
    for c, circ in aciertos:
        dibuja_caja(ax, c, IQS_VERDE, coma(circ, 3))
    for c, circ in falsas:
        dibuja_caja(ax, c, ROJO, coma(circ, 3))
    for c, circ in perdidos:
        dibuja_caja(ax, c, IQS_AZUL, 'perdido' + ('' if circ is None else f': {coma(circ, 3)}'), ls='--')
    ax.set_title(f'La escena {ESC_40 + 1} de las 40 nuevas, con ≥ {coma(CIRC_ELEGIDA)}:\n'
                 f'{len(aciertos)} VP, {len(falsas)} FP, {len(perdidos)} FN; con su circularidad', fontsize=10)
    ax.legend(handles=[plt.Line2D([], [], color=IQS_VERDE, lw=2.5, label='acierto (VP)'),
                       plt.Line2D([], [], color=ROJO, lw=2.5, label='caja falsa (FP)'),
                       plt.Line2D([], [], color=IQS_AZUL, lw=2.5, ls='--', label='disco perdido (FN)')],
              loc='upper left', bbox_to_anchor=(0, -0.03), ncol=3, fontsize=9)
    (c, circ, falsa), (recorte, ext) = CAJA_40, RECORTE_40
    ax = fig.add_subplot(gs[2])                    # el error, ampliado: la caja sin halo, para ver la pieza entera
    ax.imshow(recorte, cmap='gray', vmin=0, vmax=255, extent=ext)
    ax.add_patch(plt.Rectangle((c[0] - 0.5, c[1] - 0.5), c[2] - c[0], c[3] - c[1], fill=False, lw=1,
                               ec=ROJO if falsa else IQS_AZUL, ls='-' if falsa else '--'))
    ax.set_title(('La caja falsa' if falsa else 'El disco perdido') + ('' if circ is None else f' ({coma(circ, 3)})')
                 + (',\nampliada' if falsa else ',\nampliado'), fontsize=10)
    ax.set_xlabel('u [px]'); ax.set_ylabel('v [px]')
plt.show()

### ✎ Vuestra respuesta (dos líneas)

*Doble clic aquí y escribid, con cifras. Al acabar la clase, Archivo → Descargar → Descargar .ipynb de los dos cuadernos, el S17a y el S17b, y a la Tarea «Taller S17: procesado de imagen»: cada uno, los suyos.*

1. La circularidad mínima que elegís, y su precisión y exhaustividad en las 20 escenas y en las 40 nuevas.
2. Por qué ese valor y no otro de la banda.

## 4. Un detector aprendido: NanoDet-Plus

*En clase · diapositivas 35 y 37*

Una red de una etapa entrenada con 80 clases (OpenCV Zoo, Apache 2.0), en la CPU. Mirad sus cajas y cuánto tarda.

In [ ]:
# @title ▶ Parámetros · cambiad solo esto
# las fotos: 'el café' y 'el gato' (de scikit-image), 'la mesa' (con la luz del techo) y 'la mesa con la lámpara'
FOTOS = ['el café', 'el gato', 'la mesa']  # @param {type:"raw"}
# el umbral de confianza: las cajas por debajo se descartan
UMBRAL = 0.35  # @param {type:"number"}
# el de las fotos de la mesa, más bajo: la red no conoce nuestras piezas
UMBRAL_MESA = 0.2  # @param {type:"number"}
# el solape (IoU) por encima del cual dos cajas son la misma: se queda la más confiada
SOLAPE = 0.6  # @param {type:"number"}
# las veces que pasa cada foto por la red: la latencia es la mediana
REPETICIONES = 5  # @param {type:"number"}

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
COCO = ('persona,bicicleta,coche,moto,avión,autobús,tren,camión,barco,semáforo,boca de incendios,señal de stop,parquímetro,'
        'banco,pájaro,gato,perro,caballo,oveja,vaca,elefante,oso,cebra,jirafa,mochila,paraguas,bolso,corbata,maleta,'
        'disco volador,esquís,tabla de nieve,pelota,cometa,bate,guante,monopatín,tabla de surf,raqueta,botella,copa,taza,'
        'tenedor,cuchillo,cuchara,cuenco,plátano,manzana,bocadillo,naranja,brócoli,zanahoria,perrito caliente,pizza,rosquilla,'
        'tarta,silla,sofá,planta,cama,mesa,inodoro,televisor,portátil,ratón,mando a distancia,teclado,teléfono móvil,'
        'microondas,horno,tostadora,fregadero,nevera,libro,reloj,jarrón,tijeras,osito de peluche,secador,cepillo de dientes').split(',')
PASOS, RM, LADO = (8, 16, 32, 64), 7, 416
anclas = []
for st in PASOS:
    n = LADO//st; xv, yv = np.meshgrid(np.arange(n)*st, np.arange(n)*st)
    anclas.append(np.column_stack((xv.ravel() + 0.5*(st - 1), yv.ravel() + 0.5*(st - 1))))
MEDIA_ = np.array([103.53, 116.28, 123.675], np.float32); DESV_ = np.array([57.375, 57.12, 58.395], np.float32)
net = cv2.dnn.readNet(RUTA_DETECTOR) if RUTA_DETECTOR else None

def detectar(rgb, umbral=0.35, solape=0.6):
    """Cajas (x0, y0, x1, y1), clases y confianzas de NanoDet sobre una foto en color (RGB)."""
    bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR); h, w = bgr.shape[:2]; e = LADO/max(h, w)
    lienzo = np.zeros((LADO, LADO, 3), np.uint8); lienzo[:int(h*e), :int(w*e)] = cv2.resize(bgr, (int(w*e), int(h*e)))
    net.setInput(cv2.dnn.blobFromImage((lienzo.astype(np.float32) - MEDIA_)/DESV_))
    salidas = net.forward(net.getUnconnectedOutLayersNames())
    # se emparejan por su forma (80 clases o 32 números de caja), no por su orden: así vale en OpenCV 4 y 5
    clases = sorted([o for o in salidas if o.shape[-1] == 80], key=lambda o: -o.size)
    cajas = sorted([o for o in salidas if o.shape[-1] == 4*(RM + 1)], key=lambda o: -o.size)
    B, P = [], []
    for st, cs, bp, a in zip(PASOS, clases, cajas, anclas):
        cs, bp = cs.reshape(-1, 80), bp.reshape(-1, 4*(RM + 1))
        x = np.exp(bp.reshape(-1, RM + 1)); d = (x/x.sum(1, keepdims=True)) @ np.arange(RM + 1); d = d.reshape(-1, 4)*st
        B.append(np.column_stack([a[:, 0] - d[:, 0], a[:, 1] - d[:, 1], a[:, 0] + d[:, 2], a[:, 1] + d[:, 3]])); P.append(cs)
    B, P = np.concatenate(B)/e, np.concatenate(P)
    cl, cf = P.argmax(1), P.max(1)
    xywh = B.copy(); xywh[:, 2:] -= xywh[:, :2]
    quedan = cv2.dnn.NMSBoxes(xywh.tolist(), cf.tolist(), umbral, solape)
    return [(COCO[int(cl[i])], float(cf[i]), B[i]) for i in np.array(quedan).ravel()], int((cf >= umbral).sum())

RESPALDO = {'el café (umbral 0,35)': [('taza', 0.59), ('mesa', 0.54), ('mesa', 0.35)], 'el gato (umbral 0,35)': [('gato', 0.57)],
            'la mesa (umbral 0,2)': [('pájaro', 0.40), ('disco volador', 0.33), ('teléfono móvil', 0.22)]}
DETECCIONES = {}                                   # foto -> (imagen, umbral, latencia [s], cajas sobre el umbral, detecciones)
if net is not None:
    from skimage import data
    TODAS = {'el café': data.coffee(), 'el gato': data.chelsea(), 'la mesa': cv2.cvtColor(I_t, cv2.COLOR_GRAY2RGB),
             'la mesa con la lámpara': cv2.cvtColor(I_l, cv2.COLOR_GRAY2RGB)}
    assert all(f in TODAS for f in FOTOS), f'Las fotos son {list(TODAS)}'
    for nom in FOTOS:
        umbral = UMBRAL_MESA if nom.startswith('la mesa') else UMBRAL
        t = []
        for _ in range(int(REPETICIONES)):
            t0 = time.time(); dets, antes = detectar(TODAS[nom], umbral, SOLAPE); t.append(time.time() - t0)
        DETECCIONES[nom] = (TODAS[nom], umbral, np.median(t), antes, dets)
else:                                              # sin red, para dibujar: las fotos y las cajas guardadas (OpenCV 4.14.0)
    from skimage import data
    RESPALDO_FOTOS = {'el café': data.coffee(), 'el gato': data.chelsea(), 'la mesa': cv2.cvtColor(I_t, cv2.COLOR_GRAY2RGB)}
    RESPALDO_CAJAS = {'el café': [(165, 14, 407, 353), (1, -2, 436, 392), (137, -2, 590, 381)], 'el gato': [(16, 2, 371, 294)],
                      'la mesa': [(507, 61, 549, 98), (384, 223, 418, 257), (156, 106, 197, 150)]}   # en el orden de RESPALDO

In [ ]:
# @title ✔ Resultado
def cajas_red(ax, im, dets, ls='-'):
    # las cajas de la red sobre una foto: en verde en las fotos de scikit-image; en ámbar en la mesa, cuyas piezas no conoce
    # (en la mesa, finas y sin halo: las piezas miden unos 30 px y una caja gruesa las taparía)
    mesa = im.shape[:2] == I_t.shape
    for j, (cl, cf, b) in enumerate(dets):
        col = AMBAR if mesa else IQS_VERDE
        ax.add_patch(plt.Rectangle((b[0], b[1]), b[2] - b[0], b[3] - b[1], fill=False, ec=col, ls=ls,
                                   **(dict(lw=1.2) if mesa else dict(lw=2.2, path_effects=HALO))))
        if mesa:                                   # debajo de la caja, para no tapar la pieza
            ax.text((b[0] + b[2])/2, b[3] + 6, f'{cl} {coma(cf)}', color='white', fontsize=9, va='top', ha='center',
                    bbox=dict(fc=col, ec='none', pad=1))
        else:                                      # dentro de la caja, una debajo de otra
            ax.text(max(b[0], 0) + 4, max(b[1], 0) + 4 + 0.075*im.shape[0]*j, f'{cl} {coma(cf)}', color='white', fontsize=9,
                    va='top', bbox=dict(fc=col, ec='none', pad=1))
    ax.set_xlim(-0.5, im.shape[1] - 0.5); ax.set_ylim(im.shape[0] - 0.5, -0.5)
    if mesa:
        ax.set_title(ax.get_title() + '\nen ámbar: no conoce estas piezas', fontsize=ax.title.get_fontsize())

if net is None:
    print('Sin el detector, lo que habría salido (medido con OpenCV 4.14.0; aquí no hay red):')
    for nom, dets in RESPALDO.items():
        print(f'  {nom}: ' + '; '.join(f'{cl} {coma(cf)}' for cl, cf in dets))
    fotos = list(RESPALDO_FOTOS.values())
    fig, axs = plt.subplots(1, len(fotos), figsize=(15, 4.3), gridspec_kw=dict(width_ratios=[im.shape[1]/im.shape[0] for im in fotos]))
    for ax, (nom, dets), im, cajas in zip(axs, RESPALDO.items(), fotos, RESPALDO_CAJAS.values()):
        ax.imshow(im); ax.set_title(nom, fontsize=11); sin_ejes(ax)
        cajas_red(ax, im, [(cl, cf, b) for (cl, cf), b in zip(dets, cajas)], ls='--')
    plt.suptitle('Sin red: las cajas guardadas (OpenCV 4.14.0), en discontinua'); plt.tight_layout(rect=(0, 0, 1, 0.93)); plt.show()
elif DETECCIONES:
    fotos = list(DETECCIONES.items())
    fig = plt.figure(figsize=(15, 5.8))            # arriba, las fotos; debajo, a lo ancho, la latencia
    gs = fig.add_gridspec(2, len(fotos), width_ratios=[im.shape[1]/im.shape[0] for _, (im, *_) in fotos], height_ratios=[1, 0.35])
    axs = [fig.add_subplot(gs[0, k]) for k in range(len(fotos))] + [fig.add_subplot(gs[1, :])]
    for ax, (nom, (im, umbral, lat, antes, dets)) in zip(axs, fotos):
        print(f'{nom}: {coma(1000*lat, 0)} ms; {antes} cajas por encima de {coma(umbral, 2)} y {len(dets)} tras suprimir las repetidas:')
        for cl, cf, b in dets:
            print(f'   {cl} {coma(cf)}  [{b[0]:.0f}, {b[1]:.0f}, {b[2]:.0f}, {b[3]:.0f}]')
        ax.imshow(im); ax.set_title(f'{nom} (umbral {coma(umbral, 2)})', fontsize=11); sin_ejes(ax)
        cajas_red(ax, im, dets)
    ax = axs[-1]                                   # la latencia, frente a la de 30 fotos por segundo
    lat_ms = [1000*lat for _, (_, _, lat, _, _) in fotos]
    ax.barh(range(len(fotos)), lat_ms, color=IQS_AZUL, height=0.6)
    for k, v in enumerate(lat_ms):
        ax.text(v, k, f' {coma(v, 0)} ms', va='center', fontsize=9.5)
    ax.axvline(1000/30, color='black', lw=1.5, ls='--')
    ax.text(1000/30, -0.75, ' 30 fotos por segundo: 33 ms', fontsize=9, va='center')
    ax.set_yticks(range(len(fotos))); ax.set_yticklabels([nom for nom, _ in fotos], fontsize=9.5)
    ax.set_xlim(0, 1.6*max(lat_ms + [1000/30])); ax.set_ylim(len(fotos) - 0.5, -1.1)
    ax.set_xlabel('ms por foto'); ax.set_title(f'Latencia (mediana de {int(REPETICIONES)})', fontsize=11)
    ax.xaxis.set_major_formatter(COMA)
    plt.tight_layout(); plt.show()

**Qué tiene que salir:** taza 0,59 y mesa 0,54; gato 0,57; en la mesa, pájaro 0,40, disco volador 0,33 y teléfono móvil 0,22; unos 90 ms por foto en una CPU; en la figura, la latencia frente a los 33 ms de 30 fotos por segundo.

### Por dentro: los 24 primeros núcleos

En gris, con su parecido con Sobel.

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
def parecido(a, b):
    """La correlación de dos núcleos, sin su media."""
    a, b = a.ravel() - a.mean(), b.ravel() - b.mean(); return float(a @ b/np.linalg.norm(a)/np.linalg.norm(b))

if net is not None:
    capas = net.getLayerNames()
    TIPOS = [net.getLayer(c).type for c in capas[:3]]            # las tres primeras capas
    net.setInput(np.random.default_rng(0).normal(0, 1, (1, 3, 416, 416)).astype(np.float32))
    o_c, o_r, o_p = net.forward(list(capas[:3]))                  # lo que sale de cada una con una entrada al azar
    neg = o_c < 0
    FUGA = float(np.median(o_r[neg]/o_c[neg]))                    # por cuánto multiplica la «ReLU» lo negativo
    w1 = net.getParam(capas[0], 0)                                # la primera capa: 24 núcleos de 3 x 3 x 3 (canal, alto, ancho)
    gris = w1.sum(1)                                              # cada núcleo, visto en gris: la suma de sus tres canales
    PARECIDOS = [(parecido(g, SOBEL), parecido(g, SOBEL.T)) for g in gris]   # con Sobel en u y en v

In [ ]:
# @title ✔ Resultado
if net is None:
    print('Sin el detector no hay capas que mirar: sus cifras están en la diapositiva 35.')
else:
    print('NanoDet:', len(capas), 'capas; las tres primeras:', ', '.join(TIPOS))
    print(f'  la convolución: de 416 x 416 a {o_c.shape[-1]} x {o_c.shape[-2]} (los núcleos se aplican cada 2 píxeles);'
          f' la «ReLU» multiplica lo negativo por {coma(FUGA)} (una ReLU con fuga);'
          f' la reducción deja {o_p.shape[-1]} x {o_p.shape[-2]}')
    print('la primera capa:', w1.shape, '=', w1.size, 'pesos')
    fig, axs = plt.subplots(3, 8, figsize=(12, 4.6))
    for k, ax in enumerate(axs.ravel()):
        su, sv = PARECIDOS[k]
        s = round(max(abs(su), abs(sv)), 2)                       # el parecido, como se escribe
        lim = np.abs(gris[k]).max(); ax.imshow(gris[k], cmap='RdBu_r', vmin=-lim, vmax=lim); sin_ejes(ax)
        ax.set_title(f'{k}: {coma(s)}', fontsize=9, color=IQS_VERDE if s >= 0.85 else 'black')
    plt.suptitle('Los 24 primeros núcleos de NanoDet, en gris, y su parecido con Sobel (en u o en v); en verde, 0,85 o más')
    plt.tight_layout(); plt.show()
    print('el núcleo 11, con Sobel en u:', coma(PARECIDOS[11][0]))

**Qué tiene que salir:** 231 capas y 648 pesos en la primera; el núcleo 11, 0,93 con Sobel en u.

### Ejercicio 1 (en casa): lo que ha aprendido la neurona

Los pesos de `m1` (el techo) y de `m2` (las dos luces): ¿qué núcleo ha aprendido cada una? ¿Por qué `m2` falla menos con la lámpara?

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
E1 = []        # (con qué luz, los pesos, el umbral, el nivel de gris de corte, la suma, el peso del centro, el menor)
for e1_m, e1_t in ((m1, 'con el techo'), (m2, 'con las dos luces')):
    e1_w, e1_b = e1_m.coef_.reshape(K7, K7), e1_m.intercept_[0]
    E1.append((e1_t, e1_w, e1_b, -255*e1_b/e1_w.sum(),   # pieza si la media ponderada del vecindario pasa de este nivel
               e1_w.sum(), e1_w[K7//2, K7//2], e1_w.min()))

In [ ]:
# @title ✔ Resultado
fig, axs = plt.subplots(1, 2, figsize=(9, 3.9), layout='constrained')
lim = max(np.abs(e[1]).max() for e in E1)                  # la misma escala de color en las dos
for ax, (t, w, b, nivel, suma, centro, menor) in zip(axs, E1):
    h = ax.imshow(w, cmap='RdBu_r', vmin=-lim, vmax=lim); ax.set_title(f'{t}: {K7} × {K7} pesos'); sin_ejes(ax)
    print(f'{t}: suma de los pesos {coma(suma)}; el del centro {coma(centro)}; el menor {coma(menor)};'
          f' umbral {coma(b)}: pieza si la media ponderada pasa de {coma(nivel, 0)}')
fig.colorbar(h, ax=axs, label='peso (rojo, positivo; azul, negativo)', shrink=0.9); plt.show()

### Ejercicio 2 (en casa): un detector de cuadrados

El detector de la sección 3, buscando cuadrados en las mismas 20 escenas: ¿es tan fácil como el de discos? ¿Por qué? Probad hasta 0,86.

In [ ]:
# @title ▶ Parámetros · cambiad solo esto
# la circularidad de un cuadrado, mínima y máxima
E2_CIRC = (0.70, 0.80)  # @param {type:"raw"}
# su área en la imagen, mínima y máxima [px]
E2_AREA = (500, 1500)  # @param {type:"raw"}

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
e2_rng = np.random.default_rng(19)                 # las mismas 20 escenas de la sección 3
E2_VP = E2_FP = E2_FN = 0
E2_CIRC_CUADRADOS = []                             # la circularidad de lo que es un cuadrado de verdad
for k in range(20):
    e2_ps = escena_det(e2_rng)
    e2_I, e2_V = cam.foto(e2_ps, e2_rng, mascara=True, **(LAMPARA if k % 2 else {}))
    e2_verdad = [caja_de(e2_V, i + 1) for i, p in enumerate(e2_ps) if p.forma == 'cuadrado' and (e2_V == i + 1).sum() > 0]
    e2_b = cv2.morphologyEx(cv2.adaptiveThreshold(e2_I, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY, int(VENTANA), -MAS),
                            cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
    e2_comp = piezas_en(e2_b, E2_AREA[0], E2_AREA[1])
    E2_CIRC_CUADRADOS += [p['circ'] for p in e2_comp if any(iou(p['caja'], v) >= IOU_MIN for v in e2_verdad)]
    e2_usadas = set()
    for e2_c in [p['caja'] for p in e2_comp if E2_CIRC[0] <= p['circ'] <= E2_CIRC[1]]:
        e2_io = [(iou(e2_c, v), j) for j, v in enumerate(e2_verdad) if j not in e2_usadas]
        if e2_io and max(e2_io)[0] >= IOU_MIN:
            e2_usadas.add(max(e2_io)[1]); E2_VP += 1
        else:
            E2_FP += 1
    E2_FN += len(e2_verdad) - len(e2_usadas)
E2_PRECISION, E2_EXHAUSTIVIDAD = E2_VP/max(E2_VP + E2_FP, 1), E2_VP/max(E2_VP + E2_FN, 1)

In [ ]:
# @title ✔ Resultado
print(f'Cuadrados, circularidad de {coma(E2_CIRC[0])} a {coma(E2_CIRC[1])}: VP {E2_VP}, FP {E2_FP}, FN {E2_FN};'
      f' precisión {coma(E2_PRECISION, 3)}; exhaustividad {coma(E2_EXHAUSTIVIDAD, 3)}')
print(f'Los cuadrados de verdad miden de {coma(min(E2_CIRC_CUADRADOS))} a {coma(max(E2_CIRC_CUADRADOS))} de circularidad.')
fig, ax = plt.subplots(figsize=(9, 3))
ax.axvspan(E2_CIRC[0], E2_CIRC[1], color=IQS_VERDE, alpha=0.12, label=f'la ventana: de {coma(E2_CIRC[0])} a {coma(E2_CIRC[1])}')
c2 = np.sort(E2_CIRC_CUADRADOS); pila = {}
for c in c2:                                       # un punto por cuadrado, apilados de 0,01 en 0,01
    k = int(np.floor(c*100 + 1e-9)); pila[k] = pila.get(k, 0) + 1
    dentro = E2_CIRC[0] <= c <= E2_CIRC[1]
    ax.plot(c, pila[k], 'o', color=IQS_VERDE if dentro else IQS_AZUL, ms=9)
ax.plot([], [], 'o', color=IQS_VERDE, ms=9, label='dentro: se detecta'); ax.plot([], [], 'o', color=IQS_AZUL, ms=9, label='fuera: se pierde')
ax.set_xlim(0.6, 0.95); ax.set_ylim(0, max(pila.values()) + 1); ax.set_yticks(range(1, max(pila.values()) + 1))
ax.set_xlabel('circularidad'); ax.set_ylabel('cuadrados'); ax.grid(True, axis='x')
ax.set_title(f'La circularidad de los {len(c2)} cuadrados de verdad de las 20 escenas'); ax.legend(fontsize=9, loc='upper right')
ejes(ax); plt.tight_layout(); plt.show()

### Ejercicio 3 (en casa): la caja de la red, con la lámpara

El IoU del «disco volador» con el disco verdadero, con el techo y con la lámpara: ¿por qué baja? ¿Y si se exigiera un IoU de 0,7?

In [ ]:
# @title ▶ Parámetros · cambiad solo esto
# el umbral de confianza de NanoDet
E3_UMBRAL = 0.2  # @param {type:"number"}

In [ ]:
# @title ⚙ Cálculo · no hace falta tocarlo
E3 = []                                            # (luz, detecciones, [(confianza, caja, IoU) de cada «disco volador»])
E3_FOTOS = []                                      # para dibujar: (foto, caja del disco verdadero), con cada luz
if net is not None:
    for e3_luz, e3_I, e3_V in (('con la luz del techo', I_t, V_t), ('con la lámpara', I_l, V_l)):
        e3_dets, _ = detectar(cv2.cvtColor(e3_I, cv2.COLOR_GRAY2RGB), E3_UMBRAL)
        e3_discos = []
        for e3_cl, e3_cf, e3_b in e3_dets:
            if e3_cl == 'disco volador':
                e3_bi = [int(round(x)) for x in e3_b]                 # la caja, en píxeles enteros
                e3_discos.append((e3_cf, e3_bi, iou(e3_bi, caja_de(e3_V, 1))))
        E3.append((e3_luz, e3_dets, e3_discos))
        E3_FOTOS.append((e3_I, caja_de(e3_V, 1)))  # para dibujar: la foto y la caja del disco verdadero

In [ ]:
# @title ✔ Resultado
if net is None:
    print('Sin el detector: lo que sale está en las soluciones.')
for luz, dets, discos in E3:
    print(f'{luz}: ' + '; '.join(f'{cl} {coma(cf)}' for cl, cf, _ in dets))
    for cf, bi, io in discos:
        print(f'   disco volador {coma(cf)} {bi}: IoU {coma(io)} con el disco verdadero')
if E3:
    fig, axs = plt.subplots(1, len(E3), figsize=(11, 4.4), squeeze=False)
    for ax, (luz, dets, discos), (im, cv) in zip(axs.ravel(), E3, E3_FOTOS):
        ev0, ev1, eu0, eu1 = cv[1] - 30, cv[3] + 30, cv[0] - 45, cv[2] + 35          # el disco, ampliado
        ax.imshow(im[ev0:ev1, eu0:eu1], cmap='gray', vmin=0, vmax=255, extent=(eu0 - 0.5, eu1 - 0.5, ev1 - 0.5, ev0 - 0.5))
        dibuja_caja(ax, cv, IQS_AZUL)
        for cf, bi, io in discos:
            dibuja_caja(ax, bi, AMBAR, f'disco volador {coma(cf)}', ls='--')
        ax.set_title(f'{luz} (ampliada): IoU ' + ', '.join(coma(io) for _, _, io in discos) if discos else
                     f'{luz} (ampliada):\nningún «disco volador» con umbral {coma(E3_UMBRAL)}', fontsize=11)
        ax.set_xlabel('u [px]'); ax.set_ylabel('v [px]')
    leyenda = [plt.Line2D([], [], color=IQS_AZUL, lw=2.5, label='el disco verdadero')]
    if any(discos for _, _, discos in E3):                       # la caja de la red, solo si la hay
        leyenda.append(plt.Line2D([], [], color=AMBAR, lw=2.5, ls='--', label='la caja de la red'))
    fig.legend(handles=leyenda, loc='lower center', ncol=2, fontsize=10)
    plt.tight_layout(rect=(0, 0.07, 1, 1)); plt.show()

---

## Soluciones

**Ejercicio de clase.** Con 0,75 entran los cuadrados (0,73 a 0,84): precisión 0,678. Con 0,94 se pierden ocho discos (el que menos, 0,92): exhaustividad 0,8. Con las 20 escenas, de 0,85 a 0,92 dan las dos a 1; en las 40 nuevas, 0,85 deja pasar una caja falsa (precisión 0,989: algo que no es un disco mide 0,86) y de 0,88 a 0,92 siguen en 1 y 1. Lo sensato es 0,90, en el centro de la banda: elegido en el borde, el umbral acierta con sus escenas y falla con las nuevas.

**Ejercicio 1.** Con el techo, un suavizado y un umbral: todos los pesos positivos, el mayor en el centro (1,72); pieza si la media ponderada del vecindario pasa de 124. Con las dos luces, más peso en el centro (3,17), pesos negativos arriba y en una esquina (hasta −0,39) y corte en 103: empieza a comparar el centro con su entorno, como el umbral adaptativo, y con la lámpara acierta el 96,0 % frente al 83,7 %. Un vecindario de 7 × 7 es menor que una pieza (32 px): haría falta uno mayor o más capas (el campo receptivo, diapositiva 35).

**Ejercicio 2.** Precisión 1 y exhaustividad 0,667: 16 de 24 cuadrados. Miden de 0,73 a 0,84, según su tamaño y su giro (un borde en diagonal es una escalera de píxeles); de 0,70 a 0,86 salen los 24 sin cajas falsas (los triángulos no pasan de 0,67; los discos empiezan en 0,92). Funciona con poco margen: para cuadrados, mejor contar esquinas (sección 4 del S17a) o ajustar un polígono.

**Ejercicio 3.** Con el techo, el «disco volador» (0,33) tiene un IoU de 0,91; con la lámpara (0,23), de 0,59: la caja se estira hacia la sombra, que para la red es parte del objeto. Con un IoU mínimo de 0,7 sería una caja falsa y un disco perdido: bajan la exhaustividad y la precisión. Habría que entrenarla con nuestras fotos, con las dos luces.

---

## Para llevarse de esta sesión

- **Los mismos núcleos, aprendidos:** ajustado con ejemplos, sale casi Sobel.
- **Acertar al entrenar no es acertar:** se mide con datos nuevos y de su dominio.
- **Un detector se mide:** IoU, precisión y exhaustividad con su umbral, y su latencia.
- **Una red no conoce nuestras piezas,** y la geometría no se aprende: hace falta la K de S16.

*Cuaderno del curso 82514 Mecatrónica y Robótica · IQS Universitat Ramon Llull · curso 2026/27*

*© Guillermo Reyes Carmenaty · Publicado bajo [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/deed.es): puedes usarlo, adaptarlo y redistribuirlo, incluso con fines comerciales, siempre que reconozcas la autoría.*